# Notebook 2 · Pré-processamento

Objetivo: converter os problemas encontrados no NB_1 em regras claras e reproduzíveis. Este notebook prepara os dados para seleção de variáveis e modelação. Não escolhe o modelo final.

## O que estamos a tentar resolver?

Queremos prever se, depois de uma compra, o mesmo cliente comprará novamente nos 90 dias seguintes. O NB_1 observou os ficheiros em bruto e diagnosticou problemas; aqui transformamos esses achados em decisões que podem ser repetidas com dados novos.

**O papel deste notebook:** entregar um conjunto de variáveis utilizável e uma receita de pré-processamento. Ainda não escolhemos um modelo nem concluímos que uma variável é útil só porque parece relacionada com o alvo no NB_1.

| Observação no NB_1 | Decisão neste NB_2 | Razão |
|---|---|---|
| 65 linhas de treino têm cerca de 36 campos vazios e não têm data | Excluí-las apenas do treino | Falta quase toda a informação da compra e não podem entrar numa divisão temporal fiável |
| Existem nulos porque o cliente ainda não tinha histórico | Conservar essas compras | Ausência de histórico é uma situação real, também possível em dados novos |
| Há valores 1e9, negativos e incoerentes em duas variáveis históricas | Reconstruir a média quando a identidade é verificável; tornar o gasto anual inválido em nulo | Corrigir só quando há base para isso |
| Intervalos de dias de -4 são impossíveis | Torná-los nulos | Não sabemos o intervalo verdadeiro |
| O teste é posterior ao treino e o alvo olha 90 dias para a frente | Validação cronológica com margem de 90 dias | Aproximar a avaliação ao uso futuro e evitar que o alvo do treino alcance a validação |
| ID, customer_id, data absoluta e random_noise exigem cautela | Excluí-los das variáveis de base; guardar a data para o corte e o ID para a submissão | Evitar memorizar identificadores ou usar a data como categoria |
| A ficha de cliente cobre todas as compras, mas muitas colunas identificam o cliente | Manter uma opção de comparação restrita | Avaliar utilidade sem assumir que todos os campos são seguros ou úteis |

**Outros achados que influenciam os próximos passos:** não há compras exatamente duplicadas; várias linhas do mesmo cliente são compras diferentes. O alvo é 1 em 55,92% das linhas, o que dá um ponto de comparação simples para o NB_4. Existem distribuições assimétricas, colunas quase constantes e variáveis muito correlacionadas: serão avaliadas no NB_3/NB_4, em vez de eliminadas automaticamente. O teste contém categorias novas e clientes novos, por isso a transformação tem de os aceitar. O alvo quase se reconstrói olhando para a compra seguinte, pelo que nunca criaremos uma variável com informação posterior à compra atual.

**Regra central:** operações que aprendem valores dos dados — medianas, escalas e frequências de categorias — devem ser ajustadas só na parte de treino de cada avaliação. A validação e o teste recebem as regras já aprendidas.

**Glossário curto:** um nulo é um valor ausente; imputar é preenchê-lo segundo uma regra; fit aprende os parâmetros da regra; transform aplica-os sem voltar a aprendê-los; one-hot cria indicadores 0/1 para categorias; fuga de informação é usar no treino algo que só seria conhecido no futuro.

Ordem: ler, validar, corrigir valores conhecidos, decidir linhas utilizáveis, construir variáveis, separar treino e validação no tempo, e só então ajustar imputação, codificação e escala.

## 1. Ler e verificar

Cada linha é uma compra. ID identifica a linha de submissão, customer_id identifica o cliente e repeat_purchase_90d é o alvo.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / "data" / "Raw" / "train.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "Raw"
assert (DATA / "train.csv").exists(), "Executa a partir da raiz ou de notebooks/."

train_raw = pd.read_csv(DATA / "train.csv")
test_raw = pd.read_csv(DATA / "test.csv")
clientes = pd.read_csv(DATA / "clientes.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
print({name: df.shape for name, df in {
    "train": train_raw, "test": test_raw, "clientes": clientes, "sample": sample
}.items()})

{'train': (6534, 39), 'test': (986, 38), 'clientes': (1143, 13), 'sample': (986, 2)}


In [2]:
TARGET = "repeat_purchase_90d"
assert TARGET in train_raw and TARGET not in test_raw
assert set(train_raw.columns) - set(test_raw.columns) == {TARGET}
assert set(test_raw.columns) - set(train_raw.columns) == set()
assert train_raw["ID"].is_unique and test_raw["ID"].is_unique
assert clientes["customer_id"].is_unique
assert sample["ID"].tolist() == test_raw["ID"].tolist()
assert sample.columns.tolist() == ["ID", TARGET]
assert train_raw[TARGET].isin([0, 1]).all()
print("Estrutura, IDs e ordem da submissão: OK")

Estrutura, IDs e ordem da submissão: OK


## 2. Corrigir defeitos conhecidos

Esta etapa converte os diagnósticos do NB_1 em regras de correção. Ainda não elimina linhas nem preenche todos os valores em falta. A função `clean_fixed` trabalha sobre uma cópia, preservando os dados originais, e aplica as mesmas regras ao treino e ao teste. São regras fixas: não aprendem estatísticas a partir do teste.

### 2.1 Converter a data para permitir operações temporais

**NB_1 — secções 2 e 7.1.** `purchase_date` estava guardada como texto e aparecia inicialmente como uma categórica com 1492 valores distintos. Existiam também 65 entradas sem data válida, nas linhas praticamente vazias.

**Decisão e motivo.** Converter a coluna para `datetime`, para ordenar compras e calcular intervalos.

**Aplicação no código.** `pd.to_datetime(..., errors="coerce")` converte os valores reconhecidos e representa entradas vazias ou inválidas como `NaT` (data em falta).


### 2.2 Reconstruir médias históricas corrompidas

**NB_1 — secções 3.2 e 3.3.** A variável prior_mean_purchase_value tinha 16 valores negativos e 16 sentinelas de 1e9 no treino. O quociente prior_total_spend / prior_purchase_count reproduzia a média dentro de 1% em 98,62% das linhas avaliáveis. A nova auditoria encontrou ainda 45 médias positivas fora do intervalo entre o mínimo e o máximo das compras anteriores. Em todos esses 45 casos, a média recalculada fica dentro dos limites.

**Decisão.** Para valores negativos ou sentinelas, reconstruir a média quando há compras anteriores e gasto total não negativo. Para uma média positiva fora dos limites, reconstruí-la apenas quando o quociente fica entre o mínimo e o máximo, com 0,05 de tolerância para arredondamentos. A mesma regra encontrou mais 6 casos deste último tipo no teste. Ficou ainda uma média no treino e uma no teste que estavam dentro dos limites, mas diferiam do quociente por mais de 1%; também são reconstruídas porque o quociente cai dentro dos limites. Quando os componentes necessários não são válidos, manter o valor em falta.

**Porquê assim?** A correção usa apenas atributos da própria linha que descrevem o passado do cliente. Não olha para o alvo nem para compras posteriores. A condição dos limites impede substituir uma média suspeita por outra igualmente implausível. Médias já em falta, mas não identificadas como corrompidas, continuam em falta para a imputação posterior.

### 2.3 Marcar gastos anuais impossíveis como desconhecidos

**NB_1 — secções 3.2 e 3.3.** Em spend_365d havia 15 valores negativos e 15 sentinelas de 1e9 no treino. A auditoria com tolerância de 0,05 encontrou mais 39 valores que excedem prior_total_spend, o gasto de todo o histórico anterior. Esses 39 valores são cerca de 1,5 a 23 vezes o total histórico; não são diferenças de arredondamento. No teste, a mesma regra encontrou mais 7 casos.

**Decisão.** Marcar como nulo um gasto anual negativo, sentinela ou superior ao total histórico por mais de 0,05. Não conseguimos reconstruir o valor exato da janela de 365 dias: o gasto total pode incluir compras mais antigas, e o gasto de 180 dias não revela o resto da janela anual.

**Porquê nulo e não zero?** Zero significaria que sabemos que o cliente não gastou nada. Aqui sabemos apenas que o valor registado é impossível. A imputação posterior aprenderá uma alternativa a partir do treino e poderá conservar um indicador de ausência.

### 2.4 Marcar intervalos negativos como desconhecidos

**NB_1 — secção 3.2.** days_since_previous_purchase e prior_mean_gap_days tinham, cada uma, 16 valores negativos no treino, todos iguais a -4. No teste há 2 por coluna. Um intervalo negativo não tem significado aqui, mas a repetição não permite descobrir o valor verdadeiro.

**Decisão.** Transformar os intervalos negativos em nulos. Não usamos o valor absoluto, porque -4 não prova que o intervalo correto fosse 4 dias. Uma reconstrução a partir de datas exigiria garantir que todo o histórico necessário estava disponível; esta versão não assume isso.

### 2.5 Manter outras incoerências sem correção arbitrária

**NB_1 — secção 3.3.** A relação proposta para Total a Pagar não fecha em várias linhas e há casos de Custo superior a Total Liquido. Uma venda abaixo do custo pode ser real, e uma soma que não fecha não identifica qual componente está errado.

**Decisão.** Não forçar estas relações nem descartar todas as linhas envolvidas. A secção 3.6 volta a medir as principais relações após as correções. As diferenças que permanecerem ficam documentadas para análise e eventual comparação de modelos.

In [3]:
def clean_fixed(frame):
    out = frame.copy()
    out["purchase_date"] = pd.to_datetime(out["purchase_date"], errors="coerce")

    col = "prior_mean_purchase_value"
    invalid_mean = out[col].lt(0) | out[col].ge(1e8)
    valid_parts = out["prior_purchase_count"].gt(0) & out["prior_total_spend"].ge(0)
    reconstructed = out["prior_total_spend"] / out["prior_purchase_count"].replace(0, np.nan)
    out.loc[invalid_mean & valid_parts, col] = reconstructed[invalid_mean & valid_parts]
    out.loc[invalid_mean & ~valid_parts, col] = np.nan

    outside_bounds = (
        out[col].lt(out["prior_min_purchase_value"] - 0.05) |
        out[col].gt(out["prior_max_purchase_value"] + 0.05)
    )
    reconstructed_in_bounds = (
        reconstructed.ge(out["prior_min_purchase_value"] - 0.05) &
        reconstructed.le(out["prior_max_purchase_value"] + 0.05)
    )
    repair_bounds = outside_bounds & reconstructed_in_bounds & valid_parts
    out.loc[repair_bounds, col] = reconstructed[repair_bounds]

    identity_disagrees = (out[col] - reconstructed).abs().gt(
        np.maximum(0.05, 0.01 * reconstructed.abs())
    )
    repair_identity = identity_disagrees & reconstructed_in_bounds & valid_parts
    out.loc[repair_identity, col] = reconstructed[repair_identity]

    invalid_spend = (
        out["spend_365d"].lt(0) |
        out["spend_365d"].ge(1e8) |
        out["spend_365d"].gt(out["prior_total_spend"] + 0.05)
    )
    out["spend_365d"] = out["spend_365d"].mask(invalid_spend)

    for gap in ("days_since_previous_purchase", "prior_mean_gap_days"):
        out[gap] = out[gap].mask(out[gap].lt(0))
    return out

train_clean = clean_fixed(train_raw)
test_clean = clean_fixed(test_raw)
for name, before, after in (("train", train_raw, train_clean), ("test", test_raw, test_clean)):
    changed = {c: int((before[c].notna() & after[c].isna()).sum())
               for c in ("spend_365d", "days_since_previous_purchase", "prior_mean_gap_days")}
    repaired = int((before["prior_mean_purchase_value"].notna() &
                    after["prior_mean_purchase_value"].notna() &
                    (before["prior_mean_purchase_value"] -
                     after["prior_mean_purchase_value"]).abs().gt(0.05)).sum())
    print(name, "médias reconstruídas:", repaired, "| valores tornados em falta:", changed)

train médias reconstruídas: 78 | valores tornados em falta: {'spend_365d': 69, 'days_since_previous_purchase': 16, 'prior_mean_gap_days': 16}
test médias reconstruídas: 11 | valores tornados em falta: {'spend_365d': 11, 'days_since_previous_purchase': 2, 'prior_mean_gap_days': 2}


## 3. Escolher linhas de treino

Depois de corrigir os defeitos conhecidos, decidimos quais as compras utilizáveis para treino e avaliação temporal. A decisão distingue registos praticamente vazios de compras válidas com informação parcial; não elimina automaticamente todas as linhas com valores em falta.

### 3.1 Excluir os registos praticamente vazios

**NB_1 — secção 3.1.** As cinco colunas `Total a Pagar`, `Total Bruto`, `Total Liquido`, `Total Imp_` e `Custo` tinham 65 valores em falta cada, sempre nas mesmas 65 linhas. Essas linhas tinham 36 valores em falta por linha, num ficheiro com 39 colunas. A secção 7.1 confirmou também 65 datas ausentes ou não convertíveis, correspondentes a estes registos. Não se trata de uma compra à qual falta apenas um atributo: quase toda a descrição da compra está ausente.

**Decisão e motivo.** Excluir estas linhas do conjunto utilizável. Preencher quase toda a descrição de uma compra com valores típicos acrescentaria sobretudo informação artificial, e a ausência de data impede posicionar a compra na validação temporal. É uma escolha inicial fundamentada na qualidade das variáveis, não no valor do alvo. A taxa de recompra destas linhas foi descrita no NB_1, mas não é usada como critério de exclusão.

**Aplicação no código.** `purchase_totals` identifica os cinco valores principais e `no_purchase_info` marca as linhas onde todos estão em falta. O uso de `.all(axis=1)` significa «faltam os cinco nesta linha»; não significa «falta pelo menos um». Por isso, uma linha com apenas um total em falta não é excluída por este critério.

### 3.2 Exigir data válida e alvo conhecido

**NB_1 — secções 4 e 7.1.** O alvo do treino estava completo e continha apenas 0 e 1. A data permite ordenar as compras e separar passado e futuro, respeitando a estrutura temporal identificada no NB_1.

**Decisão e motivo.** Uma linha utilizável precisa de ter alguma informação nos cinco totais, uma data válida e um alvo conhecido. Sem data não pode entrar nesta avaliação temporal; sem alvo não pode servir para aprendizagem supervisionada nem para avaliar previsões.

**Aplicação no código.** `usable` combina as três condições com `&` («e»): `~no_purchase_info` («não tem os cinco totais vazios»), data não ausente e alvo não ausente. Assim, a regra também excluiria uma linha com data inválida mesmo que tivesse totais preenchidos. Nos dados atuais, os problemas coincidem nas mesmas 65 linhas. A validade binária do alvo já foi verificada na secção 1.

**Alcance da regra.** `usable` é uma máscara de seleção, com um valor verdadeiro ou falso por linha. Nesta célula, os registos ainda não são apagados de `train_clean`; a máscara é aplicada na construção de `X`, `y` e `dates` na secção seguinte. Manter a tabela completa permite voltar a consultar as linhas excluídas.

### 3.3 Conservar compras com histórico incompleto ou vazios pontuais

**NB_1 — secções 3.1 e 4.** 2605 das 6534 linhas (39,87%) tinham pelo menos um valor em falta. Parte desses vazios é estrutural: por exemplo, um cliente sem compras anteriores não tem intervalo desde a compra anterior nem média histórica. Outros vazios são falhas pontuais em colunas como transporte ou pagamento. As regras de disponibilidade das estatísticas históricas foram testadas no NB_1.

**Decisão e motivo.** Não eliminar estas compras apenas por terem vazios. Apagar todas as linhas incompletas perderia quase 40% do treino e retiraria casos de clientes com pouco histórico, que também precisam de previsões. 

**O que o código garante.** Não usa `dropna()` sobre todas as colunas nem exige que todas as features estejam preenchidas. Uma compra com data e alvo válidos e pelo menos um dos cinco totais disponível passa este filtro.

### 3.4 Preservar todas as linhas do teste

**NB_1 — secções 2 e 7.1.** O teste tinha 986 compras, com identificadores correspondentes aos da submissão e sem datas ausentes ou inválidas. O objetivo exige uma previsão para cada ID.

**Decisão e motivo.** Não aplicar ao teste o filtro de exclusão do treino. Mesmo uma compra de teste com informação parcial precisa de uma previsão; a preparação posterior deve lidar com os seus valores em falta.

**Aplicação no código.** Os `assert` verificam que o número de linhas do teste coincide com o da submissão e que todas as datas de teste são válidas. São verificações: se uma condição falhar, a execução para para permitir investigar o problema; não eliminam linhas. A correspondência exata e a ordem dos IDs já foram verificadas na secção 1.

In [4]:
purchase_totals = ["Total a Pagar", "Total Bruto", "Total Liquido", "Total Imp_", "Custo"]
no_purchase_info = train_clean[purchase_totals].isna().all(axis=1)
usable = ~no_purchase_info & train_clean["purchase_date"].notna() & train_clean[TARGET].notna()
print("Linhas excluídas do treino:", int((~usable).sum()))
print("Linhas de treino utilizáveis:", int(usable.sum()))
assert len(test_clean) == len(sample)
assert test_clean["purchase_date"].notna().all()

Linhas excluídas do treino: 65
Linhas de treino utilizáveis: 6469


### 3.5 Confirmar os números que motivam a exclusão

A tabela seguinte recalcula os resultados centrais do NB_1. As 65 linhas sem os cinco valores principais têm, em média, 36 campos vazios; nas restantes, a média é muito menor. Também mostramos a taxa do alvo nos dois grupos para não esconder uma possível diferença. A decisão de exclusão usa a qualidade dos dados e a ausência de data, não o valor do alvo.

A segunda tabela separa nulos originais de nulos criados pelas regras fixas. Em algumas colunas, muitos vazios são explicados por um histórico demasiado curto. Os casos adicionais de intervalos negativos passam a nulo porque não conhecemos o valor real. Estes quadros são uma auditoria, não uma imputação.

In [5]:
missing_per_row = train_raw.drop(columns=[TARGET]).isna().sum(axis=1)
summary = pd.DataFrame({
    "linhas": [int(no_purchase_info.sum()), int((~no_purchase_info).sum())],
    "média de campos vazios": [missing_per_row[no_purchase_info].mean(),
                               missing_per_row[~no_purchase_info].mean()],
    "taxa do alvo": [train_raw.loc[no_purchase_info, TARGET].mean(),
                     train_raw.loc[~no_purchase_info, TARGET].mean()],
}, index=["sem valores principais da compra", "restantes compras"])
print(summary.round(3).to_string())
print("Compras com pelo menos um nulo:", int(missing_per_row.gt(0).sum()),
      "de", len(train_raw))

history_count = train_raw["prior_purchase_count"]
history_rules = {
    "days_since_previous_purchase": history_count.eq(0),
    "prior_mean_purchase_value": history_count.eq(0),
    "prior_mean_gap_days": history_count.le(1),
    "prior_std_gap_days": history_count.le(2),
}
audit = pd.DataFrame({
    "nulos originais": [int(train_raw[col].isna().sum()) for col in history_rules],
    "nulos após regras fixas": [int(train_clean[col].isna().sum()) for col in history_rules],
    "nulos explicados por histórico curto": [
        int((train_raw[col].isna() & rule).sum())
        for col, rule in history_rules.items()
    ],
}, index=list(history_rules))
print("\nValores em falta nas colunas históricas:")
print(audit.to_string())

                                  linhas  média de campos vazios  taxa do alvo
sem valores principais da compra      65                  36.000         0.508
restantes compras                   6469                   1.156         0.560
Compras com pelo menos um nulo: 2605 de 6534

Valores em falta nas colunas históricas:
                              nulos originais  nulos após regras fixas  nulos explicados por histórico curto
days_since_previous_purchase              861                      877                                   603
prior_mean_purchase_value                 862                      862                                   603
prior_mean_gap_days                      1308                     1324                                  1048
prior_std_gap_days                       1478                     1478                                  1413


### 3.6 Verificar o que a limpeza resolveu

O NB_1 encontrou incoerências entre colunas. A limpeza da secção 2 corrigiu algumas células, mas não devemos presumir que todas as relações passaram a cumprir-se. O quadro seguinte compara as mesmas linhas utilizáveis antes e depois das regras fixas. Conta apenas linhas em que os campos necessários estão presentes; por isso, o número de linhas verificáveis pode diminuir quando um valor inválido passa a nulo.

Nas desigualdades monetárias aceitamos uma diferença até 0,05, para não contar erros de ponto flutuante como violações reais. A igualdade aproximada da média mantém a tolerância de 1% ou 0,05 usada no NB_1. A comparação não usa o alvo. Uma violação que permanece não é corrigida automaticamente: pode indicar erro numa coluna, uma definição diferente da relação proposta ou um caso legítimo, como uma venda abaixo do custo.

In [6]:
raw_usable = train_raw.loc[usable]
clean_usable = train_clean.loc[usable]

def coherence_report(frame):
    mean_rebuilt = frame["prior_total_spend"] / frame["prior_purchase_count"].replace(0, np.nan)
    close_mean = (frame["prior_mean_purchase_value"] - mean_rebuilt).abs() <= np.maximum(
        0.05, 0.01 * mean_rebuilt.abs()
    )
    checks = {
        "média histórica ~ gasto / nº de compras": (
            ["prior_mean_purchase_value", "prior_total_spend", "prior_purchase_count"],
            close_mean
        ),
        "mínimo <= média histórica <= máximo": (
            ["prior_min_purchase_value", "prior_mean_purchase_value", "prior_max_purchase_value"],
            frame["prior_min_purchase_value"].le(frame["prior_mean_purchase_value"] + 0.05) &
            frame["prior_mean_purchase_value"].le(frame["prior_max_purchase_value"] + 0.05)
        ),
        "gasto histórico >= gasto 365 dias": (
            ["prior_total_spend", "spend_365d"],
            frame["prior_total_spend"].ge(frame["spend_365d"] - 0.05)
        ),
        "gasto 30d <= 90d <= 180d <= 365d": (
            ["spend_30d", "spend_90d", "spend_180d", "spend_365d"],
            frame["spend_30d"].le(frame["spend_90d"] + 0.05) &
            frame["spend_90d"].le(frame["spend_180d"] + 0.05) &
            frame["spend_180d"].le(frame["spend_365d"] + 0.05)
        ),
    }
    rows = []
    for name, (cols, ok) in checks.items():
        valid = frame[cols].notna().all(axis=1)
        if name.startswith("média histórica"):
            valid &= frame["prior_purchase_count"].gt(0)
        rows.append({"regra": name, "linhas verificáveis": int(valid.sum()),
                     "violações": int((valid & ~ok).sum())})
    return pd.DataFrame(rows).set_index("regra")

before = coherence_report(raw_usable).add_suffix(" antes")
after = coherence_report(clean_usable).add_suffix(" depois")
print(before.join(after).to_string())

                                         linhas verificáveis antes  violações antes  linhas verificáveis depois  violações depois
regra                                                                                                                            
média histórica ~ gasto / nº de compras                       5672               78                        5672                 0
mínimo <= média histórica <= máximo                           5672               77                        5672                 0
gasto histórico >= gasto 365 dias                             6274               54                        6205                 0
gasto 30d <= 90d <= 180d <= 365d                              6274               15                        6205                 0


**Leitura dos resultados.** Com a tolerância de 0,05 para arredondamentos, havia 78 médias históricas que não correspondiam ao gasto dividido pela contagem; após as correções, há 0 entre as 5672 linhas verificáveis. A média fora do intervalo mínimo–máximo passou de 77 para 0 violações. A relação entre gasto total e gasto anual passou de 54 para 0, e a ordem das janelas de gasto de 15 para 0.

O número de linhas verificáveis com gasto anual baixou de 6274 para 6205 porque **69 valores foram marcados como desconhecidos**. Portanto, zero violações após a limpeza significa que os valores restantes respeitam estas regras; não significa que tenhamos recuperado o gasto anual verdadeiro das 69 linhas. Estas linhas permanecem no treino e serão tratadas pela imputação dentro de cada divisão de avaliação.

As outras relações do NB_1, como a soma proposta para Total a Pagar e vendas abaixo do custo, continuam documentadas sem correção automática, pois a regra correta não está demonstrada.

## 4. Escolher a informação que o modelo vai receber

Depois de limpar os dados e identificar as compras utilizáveis, construímos a tabela de **features**: as variáveis que o modelo recebe para prever se o cliente volta a comprar em 90 dias. A função `make_features()` organiza essa informação; ainda não treina um modelo.

### Usar ou não o clientes.csv?

**O que descobrimos no NB_1 — secção 9.** Todas as compras têm uma ficha de cliente correspondente e a chave `customer_id` é única nessa ficha. Por isso, podemos juntar os ficheiros sem multiplicar compras. No entanto, `customer_type` está totalmente vazia e algumas colunas, como nomes e moradas, quase identificam cada cliente individualmente.

**Decisão.** Começar sem os atributos de `clientes.csv` e comparar mais tarde se acrescentá-los melhora as previsões. O NB_1 não demonstrou ainda esse benefício num modelo.

A função permite testar uma alternativa com apenas `customer_since_year` e `country_or_region`, que descrevem o ano de registo e a localização geral do cliente:

- `make_features(train_clean)`: usa apenas a informação já presente nas compras.
- `make_features(train_clean, include_clientes=True)`: acrescenta apenas ano de registo e região geral do cliente para comparação posterior.

### Que colunas ficam fora das entradas do modelo?

A função cria uma cópia e retira as seguintes colunas dessa cópia:

| Coluna | Motivo da decisão |
|---|---|
| `ID` | Identifica a compra para a submissão, não é uma característica da compra. |
| `customer_id` | Identifica o cliente; nesta versão evitamos entregar o código diretamente ao modelo. O histórico de compras mantém-se. |
| `purchase_date` | O NB_1 mostrou que é necessária para ordenar e dividir os dados no tempo. Guardamo-la separadamente. |
| `repeat_purchase_90d` | É a resposta a prever, por isso não pode ser uma entrada do modelo. |
| `random_noise` | Fica fora da versão base com base na análise da secção 8 do NB_1. Esta escolha é provisória e pode ser confirmada em validação; o nome da coluna não prova que seja inútil. |

As colunas continuam nas tabelas limpas. Retirá-las das features não apaga os dados originais.

Por fim, a função uniformiza os vazios das colunas de texto como `np.nan`. Ainda não os preenche nem converte categorias em números: isso acontece na secção 6. Também não cria novos rácios ou estatísticas nesta versão.

### O que o código produz?

| Variável | Conteúdo |
|---|---|
| `X_all` | Features de todas as compras do treino, antes de aplicar o filtro da secção 3. |
| `X` | Features das 6469 compras utilizáveis. |
| `y` | Resposta real, 0 ou 1, dessas mesmas compras. |
| `dates` | Datas dessas mesmas compras, para dividir treino e validação no tempo. |
| `X_test` | Features das 986 compras do teste, para as quais teremos de prever a resposta. |

**Interpretação do resultado obtido.** A versão base tem **34 variáveis: 28 numéricas e 6 categóricas**. Este número não significa que todas sejam realmente importantes. O código confirma que treino e teste têm as mesmas colunas, na mesma ordem, e que todas as linhas de teste foram preservadas.


In [7]:
CLIENT_COLUMNS = ["customer_id", "customer_since_year", "country_or_region"]

def make_features(clean_frame, include_clientes=False):
    out = clean_frame.copy()
    if include_clientes:
        out = out.merge(clientes[CLIENT_COLUMNS], on="customer_id",
                        how="left", validate="m:1", sort=False)
        assert len(out) == len(clean_frame)
    X = out.drop(columns=["ID", "customer_id", "purchase_date", TARGET, "random_noise"],
                 errors="ignore")
    text_cols = X.select_dtypes(include=["str", "object", "category"]).columns
    for col in text_cols:
        X[col] = X[col].astype(object).where(X[col].notna(), np.nan)
    return X

X_all = make_features(train_clean)
X_test = make_features(test_clean)
X = X_all.loc[usable].copy()
y = train_clean.loc[usable, TARGET].astype(int)
dates = train_clean.loc[usable, "purchase_date"]
assert X.columns.tolist() == X_test.columns.tolist()
assert len(X_test) == len(test_raw)
print("Variáveis de base:", X.shape[1], "| numéricas:",
      X.select_dtypes(include="number").shape[1],
      "| categóricas:", X.select_dtypes(exclude="number").shape[1])

Variáveis de base: 34 | numéricas: 28 | categóricas: 6


### 4.1 Verificar a alternativa com clientes.csv

A ligação é feita por customer_id e deve preservar o número de compras. Aqui apenas verificamos a cobertura e a estrutura da alternativa; não treinamos um modelo nem usamos o alvo para decidir se a ficha ajuda. O conjunto de base continua sem estas duas colunas. A comparação de desempenho e a disponibilidade histórica dos atributos ficam por confirmar.

In [8]:
X_clientes_train = make_features(train_clean, include_clientes=True)
X_clientes_test = make_features(test_clean, include_clientes=True)
assert len(X_clientes_train) == len(train_clean)
assert len(X_clientes_test) == len(test_clean)
assert X_clientes_train.columns.tolist() == X_clientes_test.columns.tolist()
print("Variáveis de base:", X.shape[1],
      "| alternativa com dois atributos de clientes:", X_clientes_train.shape[1])
print("Compras de treino com ficha:", int(X_clientes_train["customer_since_year"].notna().sum()),
      "de", len(X_clientes_train))
print("Compras de teste com ficha:", int(X_clientes_test["customer_since_year"].notna().sum()),
      "de", len(X_clientes_test))

Variáveis de base: 34 | alternativa com dois atributos de clientes: 36
Compras de treino com ficha: 6534 de 6534
Compras de teste com ficha: 986 de 986


## 5. Separar treino e validação no tempo

**NB_1 — secções 7.1 e 7.3.** O treino cobre compras de 2015 a fevereiro de 2025; o teste contém compras posteriores, de maio de 2025 a outubro de 2026. O alvo pergunta se existe outra compra nos 90 dias seguintes. Uma divisão aleatória misturaria passado e futuro, dando ao modelo uma avaliação pouco semelhante ao uso real. A proximidade entre compras do mesmo cliente também exige cuidado: cerca de 94,5% das linhas de teste são de clientes vistos no treino, mas há clientes novos.

**Decisão inicial.** Ordenar as datas de treino utilizáveis e usar aproximadamente os 20% mais recentes para validação. As compras anteriores ao início da validação menos 90 dias entram no ajuste; as compras na margem de 90 dias não entram em nenhum dos dois conjuntos. Assim, a janela do alvo das últimas compras usadas para ajuste não alcança o período de validação.

O corte é demonstrativo e deve ser discutido no NB_4. Um único período pode dar uma estimativa instável; todos os modelos comparados deverão usar a mesma estratégia. A data serve para dividir, mas não entra em X como uma categoria. As 65 linhas sem data já foram excluídas apenas do treino.

In [9]:
boundary = dates.sort_values().iloc[int(0.8 * len(dates))]
purge_start = boundary - pd.Timedelta(days=90)
fit_rows = dates < purge_start
validation_rows = dates >= boundary

X_fit, y_fit = X.loc[fit_rows], y.loc[fit_rows]
X_val, y_val = X.loc[validation_rows], y.loc[validation_rows]
assert dates.loc[fit_rows].max() < purge_start
assert dates.loc[validation_rows].min() >= boundary
assert not X_fit.index.intersection(X_val.index).size

print("Início da validação:", boundary.date())
print("Última data do treino:", dates.loc[fit_rows].max().date())
print("Linhas para ajuste:", len(X_fit), "| validação:", len(X_val),
      "| margem:", int((~fit_rows & ~validation_rows).sum()))

Início da validação: 2023-06-17
Última data do treino: 2023-03-18
Linhas para ajuste: 5024 | validação: 1298 | margem: 147


## 6. Preencher nulos, codificar categorias e ajustar a escala

**Porquê só depois da separação?** A mediana, a frequência das categorias e a média/desvio usados na escala são propriedades dos dados. Se fossem calculados com validação ou teste incluídos, estes conjuntos influenciariam o treino. O NB_2 cria uma receita; o NB_4 deverá colocar esta receita dentro do Pipeline de cada classificador, sobretudo se usar validação cruzada.

**Numéricas.** SimpleImputer aprende a mediana de cada coluna em X_fit e preenche valores ausentes. A opção add_indicator cria marcas para colunas que tinham nulos no ajuste. Assim, um modelo pode distinguir um valor originalmente ausente de um valor mediano observado. StandardScaler ajusta a escala usando só X_fit e pode ajudar modelos sensíveis à magnitude, como a regressão logística. Para árvores, a escala poderá ser desligada.

**Categóricas.** Os nulos tornam-se uma categoria explícita. OneHotEncoder cria indicadores para as categorias vistas no treino, agrupa níveis com menos de 20 ocorrências e aceita categorias novas sem lançar erro. Isto é necessário porque o NB_1 encontrou níveis exclusivos do teste, incluindo commercial_zone. O limite de 20 é uma escolha inicial a avaliar, não uma exigência do enunciado.

**O que ainda não fazemos.** O NB_1 encontrou assimetria e colinearidade, mas aplicar logaritmos ou retirar variáveis antes de medir o efeito pode não ajudar. Essas escolhas serão comparadas no NB_3/NB_4. As 92 colunas transformadas obtidas nesta execução resultam do conjunto de treino usado e da codificação; não são 92 variáveis originais.

In [10]:
def make_preprocessor(X_reference, scale_numeric=True):
    numeric_cols = X_reference.select_dtypes(include="number").columns.tolist()
    category_cols = X_reference.select_dtypes(exclude="number").columns.tolist()
    numeric_steps = [
        ("impute", SimpleImputer(strategy="median", add_indicator=True,
                                 keep_empty_features=True))
    ]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    numeric_pipe = Pipeline(numeric_steps)
    category_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="__missing__",
                                 keep_empty_features=True)),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=20))
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_cols),
        ("category", category_pipe, category_cols)
    ])

preprocessor = make_preprocessor(X_fit, scale_numeric=True)
Z_fit = preprocessor.fit_transform(X_fit)
Z_val = preprocessor.transform(X_val)
Z_test = preprocessor.transform(X_test)
for name, matrix in (("treino", Z_fit), ("validação", Z_val), ("teste", Z_test)):
    values = matrix.data if hasattr(matrix, "toarray") else np.asarray(matrix)
    assert np.isfinite(values).all(), f"{name} contém valores não finitos"
    print(name, matrix.shape)
assert Z_fit.shape[1] == Z_val.shape[1] == Z_test.shape[1]
assert test_clean["ID"].tolist() == sample["ID"].tolist()

treino (5024, 92)
validação (1298, 92)
teste (986, 92)


## 7. Conclusões e decisões por confirmar

| Já implementado neste NB_2 | Ainda por decidir com validação |
|---|---|
| Corrigir médias históricas corrompidas quando é possível reconstruí-las | Se a ficha de cliente acrescenta informação útil e está disponível na data da compra |
| Marcar como nulos valores impossíveis sem reconstrução segura | Se logaritmos ou outras transformações melhoram os modelos |
| Conservar compras com nulos estruturais | Se variáveis quase constantes ou colineares devem ser retiradas |
| Excluir as 65 linhas quase vazias apenas do treino e manter todo o teste | Se um ou vários cortes temporais dão uma avaliação suficientemente estável |
| Ajustar imputação, categorias e escala apenas no treino de cada avaliação | Que métrica, variáveis e modelo têm melhor desempenho |

**Resultado confirmado nesta execução:** 6469 compras de treino utilizáveis; 5024 para ajuste, 1298 para validação e 147 na margem temporal. Todas as 986 linhas de teste permanecem. A matriz transformada tem a mesma estrutura nos três conjuntos e não contém valores não finitos.

**Nota de leitura do NB_1:** algumas células da exploração não tinham execução guardada e certas contagens diferiam entre texto e outputs. Por isso, os números decisivos para estas regras são recalculados aqui.

**Auditoria da limpeza:** foram reconstruídas 78 médias históricas no treino e 11 no teste; 69 gastos anuais do treino e 11 do teste passaram a nulo. Nas linhas ainda verificáveis, as quatro relações históricas testadas na secção 3.6 não têm violações. As restantes questões listadas acima dependem de validação e da interpretação do negócio.